# Training Script for Regular RGB and Greyscale Images

In [ ]:
# ==================== YOLO TREE DETECTION - TRAINING ====================
# This script trains YOLOv11-Large segmentation model for citrus tree detection.
# Update the BASE_DIR path below to match your local setup.

# ==================== CONFIGURATION ====================
# Update this path to your base directory
BASE_DIR = "Object Detection Pipeline"

# Paths (no need to edit)
TRAIN_IMAGES = "train/images"
VAL_IMAGES = "valid/images"
YOLO_WEIGHTS = "yolo11l-seg.pt"  # Will download if not found

# Training parameters
EPOCHS = 200
BATCH_SIZE = 8
IMG_SIZE = 640
PATIENCE = 50
CONFIDENCE = 0.35
MAX_DET = 10000

# ==================== TRAINING SCRIPT ====================
from ultralytics import YOLO
import os
import torch

# Check GPU
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Get absolute path
BASE_DIR_ABS = os.path.abspath(BASE_DIR)

# Change to base directory
os.chdir(BASE_DIR_ABS)

# Create data.yaml
print("\nCreating data.yaml...")
data_yaml = f"""
path: {BASE_DIR_ABS}
train: {TRAIN_IMAGES}
val: {VAL_IMAGES}
nc: 1
names: ['tree']
"""

with open('data.yaml', 'w') as f:
    f.write(data_yaml)
print("data.yaml created")

# Clear cache files
print("\nClearing cache files...")
cache_files = [
    "train/images.cache",
    "valid/images.cache",
    "train/images.cache.npy",
    "valid/images.cache.npy"
]
for cache in cache_files:
    if os.path.exists(cache):
        os.remove(cache)
        print(f"  Deleted {cache}")

# Count images
print("\nChecking dataset...")
train_path = os.path.join(BASE_DIR_ABS, TRAIN_IMAGES)
val_path = os.path.join(BASE_DIR_ABS, VAL_IMAGES)

if not os.path.exists(train_path):
    raise FileNotFoundError(f"Training directory not found: {train_path}")

train_count = len([f for f in os.listdir(train_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
val_count = len([f for f in os.listdir(val_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

print(f"  Training images: {train_count}")
print(f"  Validation images: {val_count}")
print(f"  Total: {train_count + val_count}")

# Load model
print("\nLoading YOLOv11-Large segmentation model...")
model = YOLO(YOLO_WEIGHTS)

# Clear GPU cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("Cleared GPU cache")

# Train
print("\nTraining...")
results = model.train(
    data='data.yaml',
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    patience=PATIENCE,
    device=0 if torch.cuda.is_available() else 'cpu',
    project=BASE_DIR_ABS,
    name='tree_training',
    exist_ok=True,
    task='segment',
    workers=2,
    amp=True,
    max_det=MAX_DET,
    conf=CONFIDENCE
)
print("Training complete!")

# Find best model
best_model_path = os.path.join(BASE_DIR_ABS, 'tree_training/weights/best.pt')
if not os.path.exists(best_model_path):
    alt_path = os.path.join(BASE_DIR_ABS, 'segment/tree_training/weights/best.pt')
    if os.path.exists(alt_path):
        best_model_path = alt_path

print(f"\nModel saved to: {best_model_path}")

# ==================== TEST THE MODEL ====================
print("\nTesting model...")
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)
    
    # Test on first validation image
    val_images = [f for f in os.listdir(val_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if val_images:
        test_img = os.path.join(val_path, val_images[0])
        print(f"Testing on: {test_img}")
        
        results = model(test_img, conf=CONFIDENCE, iou=0.45, max_det=MAX_DET, verbose=True)
        
        for result in results:
            if result.masks is not None:
                tree_count = len(result.masks)
                print(f"Detected {tree_count} trees")
                
                if tree_count == 300:
                    print("WARNING: Detected exactly 300 trees - might have hit old limit!")
                
                # Save annotated image
                annotated = result.plot()
                import cv2
                save_path = os.path.join(BASE_DIR_ABS, 'test_detection.jpg')
                cv2.imwrite(save_path, annotated)
                print(f"Saved test image to: {save_path}")

print("\nDone!")

# Normalized Brightness Function

In [1]:
#NORMALIZED CODE
# brightness_normalizer.py
import cv2
import numpy as np
from scipy.ndimage import gaussian_filter

class BrightnessNormalizer:
    def __init__(self, grid_size=5, min_detections_per_patch=3,
                 min_tree_coverage=0.1, blend_radius=0.2,
                 min_gain=0.5, max_gain=2.5):
        self.grid_size = grid_size
        self.min_detections_per_patch = min_detections_per_patch
        self.min_tree_coverage = min_tree_coverage
        self.blend_radius = blend_radius
        self.min_gain = min_gain
        self.max_gain = max_gain
        self.min_confidence_for_scoring = 0.5

    def analyze_patches_with_brightness(self, image_bgr, results):
        """Analyze patches and extract brightness for each"""
        h, w = image_bgr.shape[:2]
        patch_h = h // self.grid_size
        patch_w = w // self.grid_size

        # Convert to LAB
        lab = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2LAB)
        L_channel = lab[:,:,0]

        patch_data = {}

        for row in range(self.grid_size):
            for col in range(self.grid_size):
                # Patch boundaries
                y1 = row * patch_h
                y2 = (row + 1) * patch_h if row < self.grid_size - 1 else h
                x1 = col * patch_w
                x2 = (col + 1) * patch_w if col < self.grid_size - 1 else w

                patch_key = f"{row},{col}"
                detections_in_patch = []
                tree_mask_patch = np.zeros((y2-y1, x2-x1), dtype=np.uint8)

                # Check which detections fall in this patch
                for result in results:
                    if result.boxes is not None and result.masks is not None:
                        for i, (box, mask) in enumerate(zip(result.boxes.xyxy, result.masks.xy)):
                            center_x = (box[0] + box[2]) / 2
                            center_y = (box[1] + box[3]) / 2

                            if x1 <= center_x <= x2 and y1 <= center_y <= y2:
                                conf = float(result.boxes.conf[i])
                                detections_in_patch.append(conf)

                                # Add to patch mask
                                points = np.array(mask, dtype=np.int32)
                                points_translated = points - np.array([x1, y1])
                                cv2.fillPoly(tree_mask_patch, [points_translated], 255)

                # Calculate patch metrics
                num_detections = len(detections_in_patch)
                tree_coverage = np.sum(tree_mask_patch > 0) / (tree_mask_patch.size + 1e-6)

                # Extract brightness from tree pixels
                patch_L = L_channel[y1:y2, x1:x2]
                tree_L_values = patch_L[tree_mask_patch > 0]

                if (num_detections >= self.min_detections_per_patch and
                    tree_coverage >= self.min_tree_coverage and
                    len(tree_L_values) > 0):

                    avg_conf = np.mean(detections_in_patch)
                    score = avg_conf * np.sqrt(num_detections)
                    median_L = np.median(tree_L_values)

                    patch_data[patch_key] = {
                        'score': score,
                        'avg_conf': avg_conf,
                        'num_detections': num_detections,
                        'coverage': tree_coverage,
                        'bounds': (x1, y1, x2, y2),
                        'median_L': median_L,
                        'valid': True
                    }
                else:
                    patch_data[patch_key] = {
                        'valid': False,
                        'bounds': (x1, y1, x2, y2)
                    }

        return patch_data

    def create_smooth_gain_map(self, patch_data, target_L, image_shape):
        """Create a smooth gain map for the entire image"""
        h, w = image_shape[:2]
        gain_map = np.zeros((h, w), dtype=np.float32)
        weight_map = np.zeros((h, w), dtype=np.float32)

        patch_h = h // self.grid_size
        patch_w = w // self.grid_size

        # Calculate gain for each valid patch
        patch_gains = {}
        for patch_key, data in patch_data.items():
            if data['valid'] and 'median_L' in data:
                gain = target_L / (data['median_L'] + 1e-6)
                gain = np.clip(gain, self.min_gain, self.max_gain)
                patch_gains[patch_key] = gain

        if not patch_gains:
            return np.ones((h, w), dtype=np.float32)

        # Fill gain map with Gaussian weighted contributions
        sigma = min(patch_h, patch_w) * self.blend_radius

        for row in range(self.grid_size):
            for col in range(self.grid_size):
                patch_key = f"{row},{col}"

                if patch_key in patch_gains:
                    gain = patch_gains[patch_key]
                else:
                    # Interpolate from neighbors
                    valid_neighbors = []
                    for dr in [-1, 0, 1]:
                        for dc in [-1, 0, 1]:
                            neighbor_key = f"{row+dr},{col+dc}"
                            if neighbor_key in patch_gains:
                                valid_neighbors.append(patch_gains[neighbor_key])

                    gain = np.mean(valid_neighbors) if valid_neighbors else 1.0

                # Get patch center
                center_x = col * patch_w + patch_w // 2
                center_y = row * patch_h + patch_h // 2

                # Create 2D Gaussian
                y, x = np.ogrid[:h, :w]
                gaussian = np.exp(-((x - center_x)**2 + (y - center_y)**2) / (2 * sigma**2))

                gain_map += gain * gaussian
                weight_map += gaussian

        # Normalize by weights
        gain_map = np.divide(gain_map, weight_map, where=weight_map > 0, out=np.ones_like(gain_map))

        # Additional smoothing
        gain_map = gaussian_filter(gain_map, sigma=sigma/2)
        gain_map = np.nan_to_num(gain_map, nan=1.0)

        return gain_map

    def apply_normalization(self, image_bgr, gain_map):
        """Apply spatially varying gain to image"""
        lab = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2LAB).astype(np.float32)
        lab[:,:,0] = np.clip(lab[:,:,0] * gain_map, 0, 255)
        normalized_bgr = cv2.cvtColor(lab.astype(np.uint8), cv2.COLOR_LAB2BGR)
        return normalized_bgr

    def normalize_and_detect(self, image_path, model, conf_thresh=0.35, iou_thresh=0.45):
        """Main method: normalize image and run detection"""
        # Read image
        img_bgr = cv2.imread(image_path)

        # Run initial detection
        results = model(image_path, conf=conf_thresh, iou=iou_thresh, max_det=10000, verbose=False)

        # Analyze patches
        patch_data = self.analyze_patches_with_brightness(img_bgr, results)

        # Find best patch for target brightness
        valid_patches = {k: v for k, v in patch_data.items() if v.get('valid', False)}

        if valid_patches:
            # Find patch with highest score
            best_patch_key = max(valid_patches.keys(), key=lambda k: valid_patches[k]['score'])
            target_L = valid_patches[best_patch_key]['median_L']

            # Create gain map
            gain_map = self.create_smooth_gain_map(patch_data, target_L, img_bgr.shape)

            # Check if normalization is worth it
            gain_deviation = np.max(np.abs(gain_map - 1.0))

            if gain_deviation > 0.1:  # MIN_GAIN_THRESHOLD
                # Apply normalization
                img_normalized = self.apply_normalization(img_bgr, gain_map)

                # Run detection on normalized image
                results_normalized = model(img_normalized, conf=conf_thresh, iou=iou_thresh,
                                         max_det=10000, verbose=False)

                return img_normalized, results_normalized, gain_map

        # Return original if no normalization needed
        return img_bgr, results, None

# Ensemble Voting Function

In [2]:
from ultralytics import YOLO
import numpy as np
import cv2
import os
import torch
from datetime import datetime

class EnsembleTreeDetector:
    def __init__(self, rgb_model_path, gray_model_path, gray_images_path):
        # Load models
        print("Loading models...")
        self.rgb_model = YOLO(rgb_model_path)
        self.gray_model = YOLO(gray_model_path)

        self.gray_images_path = gray_images_path
        # Set max detections
        for model in [self.rgb_model, self.gray_model]:
            model.overrides['max_det'] = 10000

        # Initialize brightness normalizer
        self.normalizer = BrightnessNormalizer()

        # Detection parameters
        self.conf_thresh = 0.35
        self.iou_thresh = 0.45
        self.rotation_angles = [0, 90, 180, 270]

        # Ensemble parameters
        self.iou_grouping_thresh = 0.5

        # WEIGHTS FOR MODELS
        self.weights = {
            'rgb_normal': 0.333,
            'rgb_bright': 0.333,
            'gray': 0.333
        }

    def rotate_image(self, image, angle):
        """Rotate image by angle"""
        if len(image.shape) == 2:  # Grayscale
            h, w = image.shape
        else:  # RGB
            h, w = image.shape[:2]

        center = (w // 2, h // 2)
        M = cv2.getRotationMatrix2D(center, angle, 1.0)

        # Calculate new dimensions
        cos = np.abs(M[0, 0])
        sin = np.abs(M[0, 1])
        new_w = int(h * sin + w * cos)
        new_h = int(h * cos + w * sin)

        # Adjust rotation matrix
        M[0, 2] += (new_w / 2) - center[0]
        M[1, 2] += (new_h / 2) - center[1]

        rotated = cv2.warpAffine(image, M, (new_w, new_h))
        return rotated, M, (w, h)

    def rotate_box_back(self, box, M, angle, orig_size):
        """Rotate box coordinates back to original orientation"""
        # Convert box to corners
        x1, y1, x2, y2 = box
        corners = np.array([
            [x1, y1, 1],
            [x2, y1, 1],
            [x2, y2, 1],
            [x1, y2, 1]
        ]).T

        # Inverse rotation
        M_inv = cv2.invertAffineTransform(M)
        corners_orig = M_inv @ corners

        # Get new bounding box
        x_coords = corners_orig[0, :]
        y_coords = corners_orig[1, :]

        x1_new = max(0, min(x_coords))
        y1_new = max(0, min(y_coords))
        x2_new = min(orig_size[0], max(x_coords))
        y2_new = min(orig_size[1], max(y_coords))

        return [x1_new, y1_new, x2_new, y2_new]

    def rotate_mask_back(self, mask_points, M, angle, orig_size):
        """Rotate mask points back to original orientation"""
        if mask_points is None or len(mask_points) == 0:
            return None

        # Convert points to homogeneous coordinates
        points = np.array(mask_points)
        ones = np.ones((points.shape[0], 1))
        points_hom = np.hstack([points, ones]).T

        # Apply inverse transformation
        M_inv = cv2.invertAffineTransform(M)
        points_orig = M_inv @ points_hom

        # Convert back to regular coordinates
        points_orig = points_orig[:2, :].T

        return points_orig

    def run_with_rotations(self, image, model, model_name):
        """Run model on image with multiple rotations"""
        all_detections = []

        for angle in self.rotation_angles:
            if angle == 0:
                # No rotation needed
                results = model(image, conf=self.conf_thresh, iou=self.iou_thresh,
                              max_det=10000, verbose=False)
                M, orig_size = None, None
            else:
                # Rotate and detect
                rotated, M, orig_size = self.rotate_image(image, angle)
                results = model(rotated, conf=self.conf_thresh, iou=self.iou_thresh,
                              max_det=10000, verbose=False)

            # Collect detections
            for result in results:
                if result.boxes is not None:
                    for i in range(len(result.boxes)):
                        box = result.boxes.xyxy[i].cpu().numpy()

                        # Rotate box back if needed
                        if angle != 0:
                            box = self.rotate_box_back(box, M, angle, orig_size)

                        # Get mask if available
                        mask = None
                        if result.masks is not None and hasattr(result.masks, 'xy'):
                            mask = result.masks.xy[i]
                            if angle != 0:
                                mask = self.rotate_mask_back(mask, M, angle, orig_size)

                        detection = {
                            'box': box,
                            'confidence': float(result.boxes.conf[i]),
                            'mask': mask,
                            'model': model_name,
                            'angle': angle
                        }
                        all_detections.append(detection)

        return all_detections

    def nms(self, detections, iou_threshold=0.5):
        """Regular NMS - keep best, remove overlaps"""
        if not detections:
            return []

        # Sort by confidence (highest first)
        detections = sorted(detections, key=lambda x: x['confidence'], reverse=True)

        keep = []
        while detections:
            # Take the best detection
            best = detections.pop(0)
            keep.append(best)

            # Remove all detections that overlap too much with 'best'
            detections = [d for d in detections
                         if self.calculate_iou(d['box'], best['box']) < iou_threshold]

        return keep

    def calculate_iou(self, box1, box2):
        """Calculate IoU between two boxes"""
        x1 = max(box1[0], box2[0])
        y1 = max(box1[1], box2[1])
        x2 = min(box1[2], box2[2])
        y2 = min(box1[3], box2[3])

        intersection = max(0, x2 - x1) * max(0, y2 - y1)

        area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
        area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])

        union = area1 + area2 - intersection

        return intersection / union if union > 0 else 0

    def detect(self, image_path):
        """Main detection method with ensemble"""
        print(f"\nProcessing: {os.path.basename(image_path)}")

        # Read RGB image
        img_rgb = cv2.imread(image_path)

          # Get corresponding grayscale image path
        img_name = os.path.basename(image_path)
        gray_image_path = os.path.join(self.gray_images_path, img_name)
        if not os.path.exists(gray_image_path):
          print(f"  WARNING: Grayscale image not found: {gray_image_path}")
          print(f"  Skipping grayscale detection for this image")
          img_gray = None
        else:
          # Read grayscale image directly
          img_gray = cv2.imread(gray_image_path)

        # Method 1: RGB Normal with rotations (RGB image)
        print("  Running RGB normal...")
        rgb_normal_detections = self.run_with_rotations(img_rgb, self.rgb_model, 'rgb_normal')
        rgb_normal_clean = self.nms(rgb_normal_detections)
        print(f"    Found {len(rgb_normal_clean)} trees")

        # Method 2: RGB Brightness Normalized (RGB image)
        print("  Running RGB brightness normalized...")
        img_normalized, results_normalized, gain_map = self.normalizer.normalize_and_detect(
            image_path, self.rgb_model, self.conf_thresh, self.iou_thresh)

        # Run rotations on normalized RGB image
        rgb_bright_detections = self.run_with_rotations(img_normalized, self.rgb_model, 'rgb_bright')
        rgb_bright_clean = self.nms(rgb_bright_detections)
        print(f"    Found {len(rgb_bright_clean)} trees")

        # Method 3: Grayscale with rotations (GRAYSCALE image)
        print("  Running Grayscale...")
        gray_detections = self.run_with_rotations(img_gray, self.gray_model, 'gray')
        gray_clean = self.nms(gray_detections)
        print(f"    Found {len(gray_clean)} trees")

        # Ensemble voting
        print("  Performing ensemble voting...")
        all_detections = rgb_normal_clean + rgb_bright_clean + gray_clean
        final_trees = self.ensemble_vote(all_detections)
        print(f"  Final result: {len(final_trees)} trees detected")

        return final_trees, {
            'rgb_normal': len(rgb_normal_clean),
            'rgb_bright': len(rgb_bright_clean),
            'gray': len(gray_clean),
            'total_before_ensemble': len(all_detections),
            'final': len(final_trees)
        }

    def ensemble_vote(self, all_detections):
        """Perform ensemble voting on all detections"""
        # Group detections by IoU
        groups = []
        used = [False] * len(all_detections)

        for i in range(len(all_detections)):
            if used[i]:
                continue

            group = [all_detections[i]]
            used[i] = True

            for j in range(i+1, len(all_detections)):
                if used[j]:
                    continue

                # Check if overlaps with any in group
                for g in group:
                    if self.calculate_iou(all_detections[j]['box'], g['box']) > self.iou_grouping_thresh:
                        group.append(all_detections[j])
                        used[j] = True
                        break

            groups.append(group)

        # Vote on each group
        final_detections = []

        for group in groups:
            # Count models
            models_in_group = set([d['model'] for d in group])
            num_models = len(models_in_group)

            # Calculate weighted confidence
            weighted_sum = sum(d['confidence'] * self.weights[d['model']] for d in group)
            weight_sum = sum(self.weights[d['model']] for d in group)
            weighted_confidence = weighted_sum / weight_sum

            # Decision logic
            keep = False
            if num_models >= 3:  # 3 models agree
                if weighted_confidence > 0.4:
                    keep = True
            elif num_models == 2:  # 2 models agree
                if weighted_confidence > 0.6:
                    keep = True
            else:  # Only 1 model
                if weighted_confidence > 0.7:
                    keep = True

            if keep:
                # Merge group into single detection
                final_det = self.merge_group(group)
                final_detections.append(final_det)

        return final_detections

    def merge_group(self, group):
        """Merge using smart union - expand based on confidence"""
        # Start with weighted average (current approach)
        total_weight = 0
        x1_weighted = y1_weighted = x2_weighted = y2_weighted = 0

        for det in group:
            w = det['confidence'] * self.weights[det['model']]
            x1_weighted += det['box'][0] * w
            y1_weighted += det['box'][1] * w
            x2_weighted += det['box'][2] * w
            y2_weighted += det['box'][3] * w
            total_weight += w

        avg_box = [
            x1_weighted / total_weight,
            y1_weighted / total_weight,
            x2_weighted / total_weight,
            y2_weighted / total_weight
        ]

        # Expand to include high-confidence outliers
        x1_final = avg_box[0]
        y1_final = avg_box[1]
        x2_final = avg_box[2]
        y2_final = avg_box[3]

        for det in group:
            if det['confidence'] > 0.7:  # High confidence detections
                x1_final = min(x1_final, det['box'][0])
                y1_final = min(y1_final, det['box'][1])
                x2_final = max(x2_final, det['box'][2])
                y2_final = max(y2_final, det['box'][3])

        final_box = [x1_final, y1_final, x2_final, y2_final]

        # Use RGB model's mask if available (since dual is removed)
        final_mask = None

        # First choice: RGB model masks
        for det in group:
            if det['model'] in ['rgb_normal', 'rgb_bright'] and det['mask'] is not None:
                final_mask = det['mask']
                break

        # Last resort: gray model mask
        if final_mask is None:
            for det in group:
                if det['model'] == 'gray' and det['mask'] is not None:
                    final_mask = det['mask']
                    break

        avg_confidence = sum(d['confidence'] for d in group) / len(group)
        models_agreed = list(set(d['model'] for d in group))

        return {
            'box': final_box,
            'mask': final_mask,
            'confidence': avg_confidence,
            'models_agreed': models_agreed,
            'num_models': len(models_agreed)
        }

    def save_results(self, final_trees, image_path, output_dir):
        """Save detection results in YOLO format with masks"""
        # Create labels directory if it doesn't exist
        labels_dir = os.path.join(output_dir, "labels")
        os.makedirs(labels_dir, exist_ok=True)

        # Get image name without extension
        base_name = os.path.splitext(os.path.basename(image_path))[0]
        label_path = os.path.join(labels_dir, f"{base_name}.txt")

        # Get image dimensions for normalization
        img = cv2.imread(image_path)
        h, w = img.shape[:2]

        # Write YOLO format labels with masks
        with open(label_path, 'w') as f:
            for tree in final_trees:
                if tree['mask'] is not None:
                    # Save as segmentation format
                    mask = tree['mask']
                    # Normalize mask coordinates
                    mask_norm = mask / np.array([w, h])
                    # Flatten to single line
                    points = mask_norm.reshape(-1).tolist()
                    if len(points) >= 6:  # Valid polygon
                        line = "0 " + " ".join([f"{p:.6f}" for p in points])
                        f.write(line + '\n')
                else:
                    # Fall back to box format if no mask
                    box = tree['box']
                    # Convert to YOLO format (normalized center x, y, width, height)
                    x_center = ((box[0] + box[2]) / 2) / w
                    y_center = ((box[1] + box[3]) / 2) / h
                    width = (box[2] - box[0]) / w
                    height = (box[3] - box[1]) / h

                    # Ensure values are within [0, 1]
                    x_center = max(0, min(1, x_center))
                    y_center = max(0, min(1, y_center))
                    width = max(0, min(1, width))
                    height = max(0, min(1, height))

                    # Write as class 0 (tree)
                    f.write(f"0 {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}\n")

        return label_path

# Testing Codes

### RGB

In [ ]:
from pathlib import Path
import os
import shutil
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionValidator

# ==================== CONFIGURATION ====================
# Update this path to your base directory
BASE_DIR = "Object Detection Pipeline"

# Paths (no need to edit)
MODEL_PATH = f"{BASE_DIR}/Models/orange_tree_model_350images/tree_training/weights/best.pt"
TEST_IMAGES = f"{BASE_DIR}/test/images"
YAML_PATH = f"{BASE_DIR}/data.yaml"

# ==================== CLEAR CACHE ====================
cache_dir = f"{BASE_DIR}/test/labels"
for f in [f"{cache_dir}.cache", f"{cache_dir}.cache.npy"]:
    if os.path.exists(f): os.remove(f)
print("Cache cleared\n")

# ==================== DATA YAML ====================
TEST_IMAGES_ABS = os.path.abspath(TEST_IMAGES)

with open(YAML_PATH, 'w') as f:
    f.write(f"path: .\n")
    f.write(f"train: {TEST_IMAGES_ABS}\n")
    f.write(f"val: {TEST_IMAGES_ABS}\n")
    f.write(f"test: {TEST_IMAGES_ABS}\n")
    f.write(f"nc: 1\n")
    f.write(f"names: ['tree']")

print(f"Created yaml at: {YAML_PATH}")

# ==================== EVALUATE ====================
model = YOLO(MODEL_PATH)

args = dict(
    data=YAML_PATH,
    split='test',
    conf=0.35,
    iou=0.5,
    max_det=10000,
    task='detect',
    plots=False
)

validator = DetectionValidator(args=args)
results = validator(model=model.model)

print(f"\n{'='*50}")
print(f"RESULTS - RGB Model")
print(f"{'='*50}")
print(f"Precision:  {results['metrics/precision(B)']:.4f}")
print(f"Recall:     {results['metrics/recall(B)']:.4f}")
print(f"mAP@50:     {results['metrics/mAP50(B)']:.4f}")
print(f"mAP@50-95:  {results['metrics/mAP50-95(B)']:.4f}")
print(f"{'='*50}")

Cache cleared

Created yaml at: C:\Users\afifb\OneDrive\Documents\mech year 2\research\Paper Dr.Bilal\pipeline\object detection\Object Detection Pipeline\data.yaml
Ultralytics 8.3.162  Python-3.13.0 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)
YOLO11l-seg summary (fused): 203 layers, 27,585,363 parameters, 0 gradients, 141.9 GFLOPs
val: Fast image access  (ping: 0.30.1 ms, read: 17.14.6 MB/s, size: 115.4 KB)


val: Scanning C:\Users\afifb\OneDrive\Documents\mech year 2\research\Paper Dr.Bilal\pipeline\object detection\Object Detection Pipeline\test\labels... 59 images, 0 backgrounds, 1 corrupt: 100%|██████████| 59/59 [00:00<00:00, 284.94it/s]

val: C:\Users\afifb\OneDrive\Documents\mech year 2\research\Paper Dr.Bilal\pipeline\object detection\Object Detection Pipeline\test\images\DJI_20250203143308_0016_V_JPG_jpg.rf.96f833b47cd4a4d577032127733789fe.jpg: ignoring corrupt image/label: Label class 1 exceeds dataset class count 1. Possible class labels are 0-0
val: New cache created: C:\Users\afifb\OneDrive\Documents\mech year 2\research\Paper Dr.Bilal\pipeline\object detection\Object Detection Pipeline\test\labels.cache



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:06<00:00,  1.54s/it]

                   all         58      26132      0.945      0.944      0.928      0.356
Speed: 0.3ms preprocess, 24.1ms inference, 0.0ms loss, 17.3ms postprocess per image

RESULTS
Precision:  0.9450
Recall:     0.9436
mAP@50:     0.9279
mAP@50-95:  0.3562


### Normalized

In [ ]:
from pathlib import Path
import os
import shutil
import cv2
import numpy as np
from scipy.ndimage import gaussian_filter
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionValidator

# ==================== CONFIGURATION ====================
# Update this path to your base directory
BASE_DIR = "Object Detection Pipeline"

# Paths (no need to edit)
MODEL_PATH = os.path.join(BASE_DIR, r"Models\orange_tree_model_350images\tree_training\weights\best.pt")
IMAGES_DIR = os.path.join(BASE_DIR, r"test\images")
LABELS_DIR = os.path.join(BASE_DIR, r"test\labels")
TEMP_DIR = os.path.join(BASE_DIR, "test_norm_temp")
YAML_PATH = os.path.join(BASE_DIR, "data_norm.yaml")

# ==================== CLEAR CACHE ====================
cache_files = [
    os.path.join(BASE_DIR, r"test\labels.cache"),
    os.path.join(BASE_DIR, r"test\labels.cache.npy"),
    os.path.join(TEMP_DIR, "labels.cache"),
    os.path.join(TEMP_DIR, "labels.cache.npy")
]
for f in cache_files:
    if os.path.exists(f): os.remove(f)
print("Cache cleared\n")

# ==================== CREATE TEMP FOLDER STRUCTURE ====================
if os.path.exists(TEMP_DIR):
    shutil.rmtree(TEMP_DIR)
os.makedirs(f"{TEMP_DIR}/images")
os.makedirs(f"{TEMP_DIR}/labels")

# Copy labels
for label_file in Path(LABELS_DIR).glob("*.txt"):
    shutil.copy(label_file, f"{TEMP_DIR}/labels/{label_file.name}")
print(f"Copied labels to {TEMP_DIR}/labels/\n")

# ==================== LOAD MODEL ====================
print("Loading model...")
model = YOLO(MODEL_PATH)

# ==================== BRIGHTNESS NORMALIZER FUNCTIONS ====================
GRID_SIZE = 5
MIN_DETECTIONS_PER_PATCH = 3
MIN_TREE_COVERAGE = 0.1
BLEND_RADIUS = 0.2
MIN_GAIN_THRESHOLD = 0.1
MIN_GAIN = 0.5
MAX_GAIN = 2.5

def analyze_patches_with_brightness(image_bgr, results, grid_size=5):
    h, w = image_bgr.shape[:2]
    patch_h = h // grid_size
    patch_w = w // grid_size
    lab = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2LAB)
    L_channel = lab[:,:,0]
    patch_data = {}

    for row in range(grid_size):
        for col in range(grid_size):
            y1 = row * patch_h
            y2 = (row + 1) * patch_h if row < grid_size - 1 else h
            x1 = col * patch_w
            x2 = (col + 1) * patch_w if col < grid_size - 1 else w

            patch_key = f"{row},{col}"
            detections_in_patch = []
            tree_mask_patch = np.zeros((y2-y1, x2-x1), dtype=np.uint8)

            for result in results:
                if result.boxes is not None and result.masks is not None:
                    for i, (box, mask) in enumerate(zip(result.boxes.xyxy, result.masks.xy)):
                        center_x = (box[0] + box[2]) / 2
                        center_y = (box[1] + box[3]) / 2

                        if x1 <= center_x <= x2 and y1 <= center_y <= y2:
                            conf = float(result.boxes.conf[i])
                            detections_in_patch.append(conf)
                            points = np.array(mask, dtype=np.int32)
                            points_translated = points - np.array([x1, y1])
                            cv2.fillPoly(tree_mask_patch, [points_translated], 255)

            num_detections = len(detections_in_patch)
            tree_coverage = np.sum(tree_mask_patch > 0) / (tree_mask_patch.size + 1e-6)
            patch_L = L_channel[y1:y2, x1:x2]
            tree_L_values = patch_L[tree_mask_patch > 0]

            if num_detections >= MIN_DETECTIONS_PER_PATCH and tree_coverage >= MIN_TREE_COVERAGE and len(tree_L_values) > 0:
                avg_conf = np.mean(detections_in_patch)
                score = avg_conf * np.sqrt(num_detections)
                median_L = np.median(tree_L_values)

                patch_data[patch_key] = {
                    'score': score,
                    'median_L': median_L,
                    'valid': True
                }
            else:
                patch_data[patch_key] = {'valid': False}
    return patch_data

def create_smooth_gain_map(patch_data, target_L, image_shape, grid_size=5):
    h, w = image_shape[:2]
    gain_map = np.zeros((h, w), dtype=np.float32)
    weight_map = np.zeros((h, w), dtype=np.float32)
    patch_h = h // grid_size
    patch_w = w // grid_size

    patch_gains = {}
    for patch_key, data in patch_data.items():
        if data['valid'] and 'median_L' in data:
            gain = target_L / (data['median_L'] + 1e-6)
            gain = np.clip(gain, MIN_GAIN, MAX_GAIN)
            patch_gains[patch_key] = gain

    if not patch_gains:
        return np.ones((h, w), dtype=np.float32)

    sigma = min(patch_h, patch_w) * BLEND_RADIUS

    for row in range(grid_size):
        for col in range(grid_size):
            patch_key = f"{row},{col}"

            if patch_key in patch_gains:
                gain = patch_gains[patch_key]
            else:
                valid_neighbors = []
                for dr in [-1, 0, 1]:
                    for dc in [-1, 0, 1]:
                        neighbor_key = f"{row+dr},{col+dc}"
                        if neighbor_key in patch_gains:
                            valid_neighbors.append(patch_gains[neighbor_key])
                gain = np.mean(valid_neighbors) if valid_neighbors else 1.0

            center_x = col * patch_w + patch_w // 2
            center_y = row * patch_h + patch_h // 2
            y, x = np.ogrid[:h, :w]
            gaussian = np.exp(-((x - center_x)**2 + (y - center_y)**2) / (2 * sigma**2))
            gain_map += gain * gaussian
            weight_map += gaussian

    gain_map = np.divide(gain_map, weight_map, where=weight_map > 0, out=np.ones_like(gain_map))
    gain_map = gaussian_filter(gain_map, sigma=sigma/2)
    gain_map = np.nan_to_num(gain_map, nan=1.0)
    return gain_map

def apply_patch_normalization(image_bgr, gain_map):
    lab = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2LAB).astype(np.float32)
    lab[:,:,0] = np.clip(lab[:,:,0] * gain_map, 0, 255)
    return cv2.cvtColor(lab.astype(np.uint8), cv2.COLOR_LAB2BGR)

# ==================== NORMALIZE ALL IMAGES ====================
print("Normalizing images...")
images = list(Path(IMAGES_DIR).glob("*.[jJ][pP][gG]")) + list(Path(IMAGES_DIR).glob("*.[pP][nN][gG]"))

for i, img_path in enumerate(images):
    img_bgr = cv2.imread(str(img_path))

    results = model(str(img_path), conf=0.35, iou=0.45, max_det=10000, verbose=False)
    patch_data = analyze_patches_with_brightness(img_bgr, results, GRID_SIZE)
    valid_patches = {k: v for k, v in patch_data.items() if v.get('valid', False)}

    if valid_patches:
        best_patch_key = max(valid_patches.keys(), key=lambda k: valid_patches[k]['score'])
        target_L = valid_patches[best_patch_key]['median_L']
        gain_map = create_smooth_gain_map(patch_data, target_L, img_bgr.shape, GRID_SIZE)

        if np.max(np.abs(gain_map - 1.0)) > MIN_GAIN_THRESHOLD:
            img_normalized = apply_patch_normalization(img_bgr, gain_map)
        else:
            img_normalized = img_bgr
    else:
        img_normalized = img_bgr

    cv2.imwrite(f"{TEMP_DIR}/images/{img_path.name}", img_normalized)

    if (i+1) % 10 == 0:
        print(f"  Normalized {i+1}/{len(images)}")

print(f"Normalized {len(images)} images\n")

# ==================== CREATE DATA YAML ====================
TEMP_DIR_ABS = os.path.abspath(TEMP_DIR)
TEMP_IMAGES_ABS = os.path.join(TEMP_DIR_ABS, "images")

with open(YAML_PATH, 'w') as f:
    f.write(f"path: .\n")
    f.write(f"train: {TEMP_IMAGES_ABS}\n")
    f.write(f"val: {TEMP_IMAGES_ABS}\n")
    f.write(f"test: {TEMP_IMAGES_ABS}\n")
    f.write(f"nc: 1\n")
    f.write(f"names: ['tree']")

print(f"Created yaml at: {YAML_PATH}")

# ==================== EVALUATE ====================
print("\nEvaluating...")
args = dict(
    data=YAML_PATH,
    split='test',
    conf=0.35,
    iou=0.5,
    max_det=10000,
    task='detect',
    plots=False
)

validator = DetectionValidator(args=args)
results = validator(model=model.model)

print(f"\n{'='*50}")
print(f"RESULTS - Brightness Normalized")
print(f"{'='*50}")
print(f"Precision:  {results['metrics/precision(B)']:.4f}")
print(f"Recall:     {results['metrics/recall(B)']:.4f}")
print(f"mAP@50:     {results['metrics/mAP50(B)']:.4f}")
print(f"mAP@50-95:  {results['metrics/mAP50-95(B)']:.4f}")
print(f"{'='*50}")

# ==================== CLEANUP ====================
shutil.rmtree(TEMP_DIR)
print("Temp folder cleaned")

Cache cleared

Copied labels to C:\Users\afifb\OneDrive\Documents\mech year 2\research\Paper Dr.Bilal\pipeline\object detection\Object Detection Pipeline\test_norm_temp/labels/

Loading model...
Normalizing images...
  Normalized 10/59
  Normalized 20/59
  Normalized 30/59
  Normalized 40/59
  Normalized 50/59
Normalized 59 images

Created yaml at: C:\Users\afifb\OneDrive\Documents\mech year 2\research\Paper Dr.Bilal\pipeline\object detection\Object Detection Pipeline\data_norm.yaml

Evaluating...
Ultralytics 8.3.162  Python-3.13.0 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)
val: Fast image access  (ping: 0.10.0 ms, read: 24.74.4 MB/s, size: 226.9 KB)


val: Scanning C:\Users\afifb\OneDrive\Documents\mech year 2\research\Paper Dr.Bilal\pipeline\object detection\Object Detection Pipeline\test_norm_temp\labels... 59 images, 0 backgrounds, 1 corrupt: 100%|██████████| 59/59 [00:00<00:00, 305.64it/s]

val: C:\Users\afifb\OneDrive\Documents\mech year 2\research\Paper Dr.Bilal\pipeline\object detection\Object Detection Pipeline\test_norm_temp\images\DJI_20250203143308_0016_V_JPG_jpg.rf.96f833b47cd4a4d577032127733789fe.jpg: ignoring corrupt image/label: Label class 1 exceeds dataset class count 1. Possible class labels are 0-0


val: New cache created: C:\Users\afifb\OneDrive\Documents\mech year 2\research\Paper Dr.Bilal\pipeline\object detection\Object Detection Pipeline\test_norm_temp\labels.cache


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:26<00:00,  6.71s/it]

                   all         58      26132      0.944      0.942      0.926      0.356
Speed: 2.4ms preprocess, 339.2ms inference, 0.0ms loss, 16.9ms postprocess per image

RESULTS - Brightness Normalized
Precision:  0.9442
Recall:     0.9423
mAP@50:     0.9260
mAP@50-95:  0.3557
Temp folder cleaned


### Grey ( +image converting)

In [13]:
from pathlib import Path
import os
import shutil
import cv2
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionValidator

# ==================== CONFIGURATION ====================
# Update this path to your base directory
BASE_DIR = "Object Detection Pipeline"

# Paths (no need to edit)
MODEL_PATH = f"{BASE_DIR}/Models/orange_tree_model_350images_greyscale/tree_training/weights/best.pt"
IMAGES_DIR = f"{BASE_DIR}/test/images"
LABELS_DIR = f"{BASE_DIR}/test/labels"
TEMP_DIR = f"{BASE_DIR}/test_grey_temp"
YAML_PATH = f"{BASE_DIR}/data_grey.yaml"

# ==================== CLEAR CACHE ====================
cache_files = [
    f"{BASE_DIR}/test/labels.cache",
    f"{BASE_DIR}/test/labels.cache.npy",
    f"{TEMP_DIR}/labels.cache",
    f"{TEMP_DIR}/labels.cache.npy"
]
for f in cache_files:
    if os.path.exists(f): os.remove(f)
print("Cache cleared\n")

# ==================== CREATE TEMP FOLDER ====================
if os.path.exists(TEMP_DIR):
    shutil.rmtree(TEMP_DIR)
os.makedirs(f"{TEMP_DIR}/images")
os.makedirs(f"{TEMP_DIR}/labels")

# Copy labels
for label_file in Path(LABELS_DIR).glob("*.txt"):
    shutil.copy(label_file, f"{TEMP_DIR}/labels/{label_file.name}")
print(f"Copied labels\n")

# ==================== CONVERT TO GREYSCALE ====================
print("Converting to greyscale...")
images = list(Path(IMAGES_DIR).glob("*.[jJ][pP][gG]")) + list(Path(IMAGES_DIR).glob("*.[pP][nN][gG]"))

for i, img_path in enumerate(images):
    img = cv2.imread(str(img_path))
    grey = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    grey_3ch = cv2.cvtColor(grey, cv2.COLOR_GRAY2BGR)  # Back to 3 channel for YOLO
    cv2.imwrite(f"{TEMP_DIR}/images/{img_path.name}", grey_3ch)

    if (i+1) % 10 == 0:
        print(f"  Converted {i+1}/{len(images)}")

print(f"Converted {len(images)} images\n")

# ==================== DATA YAML ====================
TEMP_IMAGES_ABS = os.path.abspath(f"{TEMP_DIR}/images")

with open(YAML_PATH, 'w') as f:
    f.write(f"path: .\n")
    f.write(f"train: {TEMP_IMAGES_ABS}\n")
    f.write(f"val: {TEMP_IMAGES_ABS}\n")
    f.write(f"test: {TEMP_IMAGES_ABS}\n")
    f.write(f"nc: 1\n")
    f.write(f"names: ['tree']")

print(f"Created yaml at: {YAML_PATH}")

# ==================== EVALUATE ====================
print("\nLoading model...")
model = YOLO(MODEL_PATH)

print("Evaluating...")
args = dict(
    data=YAML_PATH,
    split='test',
    conf=0.35,
    iou=0.5,
    max_det=10000,
    task='detect',
    plots=False
)

validator = DetectionValidator(args=args)
results = validator(model=model.model)

print(f"\n{'='*50}")
print(f"RESULTS - Greyscale Model")
print(f"{'='*50}")
print(f"Precision:  {results['metrics/precision(B)']:.4f}")
print(f"Recall:     {results['metrics/recall(B)']:.4f}")
print(f"mAP@50:     {results['metrics/mAP50(B)']:.4f}")
print(f"mAP@50-95:  {results['metrics/mAP50-95(B)']:.4f}")
print(f"{'='*50}")

# ==================== CLEANUP ====================
shutil.rmtree(TEMP_DIR)
print("Temp folder cleaned")

Cache cleared

Copied labels

Converting to greyscale...
  Converted 10/59
  Converted 20/59
  Converted 30/59
  Converted 40/59
  Converted 50/59
Converted 59 images

Created yaml at: Object Detection Pipeline/data_grey.yaml

Loading model...
Evaluating...
Ultralytics 8.3.162  Python-3.13.0 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)
YOLO11l-seg summary (fused): 203 layers, 27,585,363 parameters, 0 gradients, 141.9 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 25.93.8 MB/s, size: 218.6 KB)


val: Scanning C:\Users\afifb\OneDrive\Documents\mech year 2\research\Paper Dr.Bilal\pipeline\object detection\Object Detection Pipeline\test_grey_temp\labels... 59 images, 0 backgrounds, 1 corrupt: 100%|██████████| 59/59 [00:00<00:00, 306.42it/s]

val: C:\Users\afifb\OneDrive\Documents\mech year 2\research\Paper Dr.Bilal\pipeline\object detection\Object Detection Pipeline\test_grey_temp\images\DJI_20250203143308_0016_V_JPG_jpg.rf.96f833b47cd4a4d577032127733789fe.jpg: ignoring corrupt image/label: Label class 1 exceeds dataset class count 1. Possible class labels are 0-0


val: New cache created: C:\Users\afifb\OneDrive\Documents\mech year 2\research\Paper Dr.Bilal\pipeline\object detection\Object Detection Pipeline\test_grey_temp\labels.cache


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:34<00:00,  8.54s/it]

                   all         58      26132      0.951      0.933      0.931      0.373
Speed: 2.0ms preprocess, 429.4ms inference, 0.0ms loss, 39.6ms postprocess per image

RESULTS - Greyscale Model
Precision:  0.9514
Recall:     0.9327
mAP@50:     0.9309
mAP@50-95:  0.3730
Temp folder cleaned


### Ensemble (Manual Evaluation According to COCO Standards)

In [ ]:
from pathlib import Path
import os
import shutil
import cv2
import numpy as np
from scipy.ndimage import gaussian_filter
from ultralytics import YOLO

# ==================== CONFIGURATION ====================
# Update this path to your base directory
BASE_DIR = "Object Detection Pipeline"

# Paths (no need to edit)
RGB_MODEL_PATH = f"{BASE_DIR}/Models/orange_tree_model_350images/tree_training/weights/best.pt"
GRAY_MODEL_PATH = f"{BASE_DIR}/Models/orange_tree_model_350images_greyscale/tree_training/weights/best.pt"
IMAGES_DIR = f"{BASE_DIR}/test/images"
LABELS_DIR = f"{BASE_DIR}/test/labels"
GRAY_IMAGES_DIR = f"{BASE_DIR}/test/images_grey"

# ==================== CREATE GREYSCALE IMAGES ====================
print("Creating greyscale images...")
if os.path.exists(GRAY_IMAGES_DIR):
    shutil.rmtree(GRAY_IMAGES_DIR)
os.makedirs(GRAY_IMAGES_DIR)

images = list(Path(IMAGES_DIR).glob("*.[jJ][pP][gG]")) + list(Path(IMAGES_DIR).glob("*.[pP][nN][gG]"))
for img_path in images:
    img = cv2.imread(str(img_path))
    grey = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    grey_3ch = cv2.cvtColor(grey, cv2.COLOR_GRAY2BGR)
    cv2.imwrite(f"{GRAY_IMAGES_DIR}/{img_path.name}", grey_3ch)
print(f"Created {len(images)} greyscale images\n")

# ==================== BRIGHTNESS NORMALIZER CLASS ====================
class BrightnessNormalizer:
    def __init__(self, grid_size=5, min_detections_per_patch=3,
                 min_tree_coverage=0.1, blend_radius=0.2,
                 min_gain=0.5, max_gain=2.5):
        self.grid_size = grid_size
        self.min_detections_per_patch = min_detections_per_patch
        self.min_tree_coverage = min_tree_coverage
        self.blend_radius = blend_radius
        self.min_gain = min_gain
        self.max_gain = max_gain
        self.min_confidence_for_scoring = 0.5

    def analyze_patches_with_brightness(self, image_bgr, results):
        h, w = image_bgr.shape[:2]
        patch_h = h // self.grid_size
        patch_w = w // self.grid_size
        lab = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2LAB)
        L_channel = lab[:,:,0]
        patch_data = {}

        for row in range(self.grid_size):
            for col in range(self.grid_size):
                y1 = row * patch_h
                y2 = (row + 1) * patch_h if row < self.grid_size - 1 else h
                x1 = col * patch_w
                x2 = (col + 1) * patch_w if col < self.grid_size - 1 else w

                patch_key = f"{row},{col}"
                detections_in_patch = []
                tree_mask_patch = np.zeros((y2-y1, x2-x1), dtype=np.uint8)

                for result in results:
                    if result.boxes is not None and result.masks is not None:
                        for i, (box, mask) in enumerate(zip(result.boxes.xyxy, result.masks.xy)):
                            center_x = (box[0] + box[2]) / 2
                            center_y = (box[1] + box[3]) / 2

                            if x1 <= center_x <= x2 and y1 <= center_y <= y2:
                                conf = float(result.boxes.conf[i])
                                detections_in_patch.append(conf)
                                points = np.array(mask, dtype=np.int32)
                                points_translated = points - np.array([x1, y1])
                                cv2.fillPoly(tree_mask_patch, [points_translated], 255)

                num_detections = len(detections_in_patch)
                tree_coverage = np.sum(tree_mask_patch > 0) / (tree_mask_patch.size + 1e-6)
                patch_L = L_channel[y1:y2, x1:x2]
                tree_L_values = patch_L[tree_mask_patch > 0]

                if (num_detections >= self.min_detections_per_patch and
                    tree_coverage >= self.min_tree_coverage and
                    len(tree_L_values) > 0):
                    avg_conf = np.mean(detections_in_patch)
                    score = avg_conf * np.sqrt(num_detections)
                    median_L = np.median(tree_L_values)
                    patch_data[patch_key] = {
                        'score': score, 'avg_conf': avg_conf, 'num_detections': num_detections,
                        'coverage': tree_coverage, 'bounds': (x1, y1, x2, y2),
                        'median_L': median_L, 'valid': True
                    }
                else:
                    patch_data[patch_key] = {'valid': False, 'bounds': (x1, y1, x2, y2)}
        return patch_data

    def create_smooth_gain_map(self, patch_data, target_L, image_shape):
        h, w = image_shape[:2]
        gain_map = np.zeros((h, w), dtype=np.float32)
        weight_map = np.zeros((h, w), dtype=np.float32)
        patch_h = h // self.grid_size
        patch_w = w // self.grid_size

        patch_gains = {}
        for patch_key, data in patch_data.items():
            if data['valid'] and 'median_L' in data:
                gain = target_L / (data['median_L'] + 1e-6)
                gain = np.clip(gain, self.min_gain, self.max_gain)
                patch_gains[patch_key] = gain

        if not patch_gains:
            return np.ones((h, w), dtype=np.float32)

        sigma = min(patch_h, patch_w) * self.blend_radius

        for row in range(self.grid_size):
            for col in range(self.grid_size):
                patch_key = f"{row},{col}"
                if patch_key in patch_gains:
                    gain = patch_gains[patch_key]
                else:
                    valid_neighbors = []
                    for dr in [-1, 0, 1]:
                        for dc in [-1, 0, 1]:
                            neighbor_key = f"{row+dr},{col+dc}"
                            if neighbor_key in patch_gains:
                                valid_neighbors.append(patch_gains[neighbor_key])
                    gain = np.mean(valid_neighbors) if valid_neighbors else 1.0

                center_x = col * patch_w + patch_w // 2
                center_y = row * patch_h + patch_h // 2
                y, x = np.ogrid[:h, :w]
                gaussian = np.exp(-((x - center_x)**2 + (y - center_y)**2) / (2 * sigma**2))
                gain_map += gain * gaussian
                weight_map += gaussian

        gain_map = np.divide(gain_map, weight_map, where=weight_map > 0, out=np.ones_like(gain_map))
        gain_map = gaussian_filter(gain_map, sigma=sigma/2)
        gain_map = np.nan_to_num(gain_map, nan=1.0)
        return gain_map

    def apply_normalization(self, image_bgr, gain_map):
        lab = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2LAB).astype(np.float32)
        lab[:,:,0] = np.clip(lab[:,:,0] * gain_map, 0, 255)
        return cv2.cvtColor(lab.astype(np.uint8), cv2.COLOR_LAB2BGR)

    def normalize_and_detect(self, image_path, model, conf_thresh=0.35, iou_thresh=0.45):
        img_bgr = cv2.imread(image_path)
        results = model(image_path, conf=conf_thresh, iou=iou_thresh, max_det=10000, verbose=False)
        patch_data = self.analyze_patches_with_brightness(img_bgr, results)
        valid_patches = {k: v for k, v in patch_data.items() if v.get('valid', False)}

        if valid_patches:
            best_patch_key = max(valid_patches.keys(), key=lambda k: valid_patches[k]['score'])
            target_L = valid_patches[best_patch_key]['median_L']
            gain_map = self.create_smooth_gain_map(patch_data, target_L, img_bgr.shape)
            gain_deviation = np.max(np.abs(gain_map - 1.0))

            if gain_deviation > 0.1:
                img_normalized = self.apply_normalization(img_bgr, gain_map)
                results_normalized = model(img_normalized, conf=conf_thresh, iou=iou_thresh, max_det=10000, verbose=False)
                return img_normalized, results_normalized, gain_map

        return img_bgr, results, None


# ==================== ENSEMBLE DETECTOR CLASS ====================
class EnsembleTreeDetector:
    def __init__(self, rgb_model_path, gray_model_path, gray_images_path):
        print("Loading models...")
        self.rgb_model = YOLO(rgb_model_path)
        self.gray_model = YOLO(gray_model_path)
        self.gray_images_path = gray_images_path
        for model in [self.rgb_model, self.gray_model]:
            model.overrides['max_det'] = 10000
        self.normalizer = BrightnessNormalizer()
        self.conf_thresh = 0.35
        self.iou_thresh = 0.45
        self.rotation_angles = [0, 90, 180, 270]
        self.iou_grouping_thresh = 0.5
        self.weights = {'rgb_normal': 0.333, 'rgb_bright': 0.333, 'gray': 0.333}

    def rotate_image(self, image, angle):
        if len(image.shape) == 2:
            h, w = image.shape
        else:
            h, w = image.shape[:2]
        center = (w // 2, h // 2)
        M = cv2.getRotationMatrix2D(center, angle, 1.0)
        cos, sin = np.abs(M[0, 0]), np.abs(M[0, 1])
        new_w, new_h = int(h * sin + w * cos), int(h * cos + w * sin)
        M[0, 2] += (new_w / 2) - center[0]
        M[1, 2] += (new_h / 2) - center[1]
        return cv2.warpAffine(image, M, (new_w, new_h)), M, (w, h)

    def rotate_box_back(self, box, M, angle, orig_size):
        x1, y1, x2, y2 = box
        corners = np.array([[x1, y1, 1], [x2, y1, 1], [x2, y2, 1], [x1, y2, 1]]).T
        M_inv = cv2.invertAffineTransform(M)
        corners_orig = M_inv @ corners
        return [max(0, min(corners_orig[0])), max(0, min(corners_orig[1])),
                min(orig_size[0], max(corners_orig[0])), min(orig_size[1], max(corners_orig[1]))]

    def run_with_rotations(self, image, model, model_name):
        all_detections = []
        for angle in self.rotation_angles:
            if angle == 0:
                results = model(image, conf=self.conf_thresh, iou=self.iou_thresh, max_det=10000, verbose=False)
                M, orig_size = None, None
            else:
                rotated, M, orig_size = self.rotate_image(image, angle)
                results = model(rotated, conf=self.conf_thresh, iou=self.iou_thresh, max_det=10000, verbose=False)

            for result in results:
                if result.boxes is not None:
                    for i in range(len(result.boxes)):
                        box = result.boxes.xyxy[i].cpu().numpy()
                        if angle != 0:
                            box = self.rotate_box_back(box, M, angle, orig_size)
                        all_detections.append({
                            'box': box, 'confidence': float(result.boxes.conf[i]),
                            'model': model_name, 'angle': angle
                        })
        return all_detections

    def nms(self, detections, iou_threshold=0.5):
        if not detections:
            return []
        detections = sorted(detections, key=lambda x: x['confidence'], reverse=True)
        keep = []
        while detections:
            best = detections.pop(0)
            keep.append(best)
            detections = [d for d in detections if self.calculate_iou(d['box'], best['box']) < iou_threshold]
        return keep

    def calculate_iou(self, box1, box2):
        x1, y1 = max(box1[0], box2[0]), max(box1[1], box2[1])
        x2, y2 = min(box1[2], box2[2]), min(box1[3], box2[3])
        inter = max(0, x2 - x1) * max(0, y2 - y1)
        area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
        area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
        union = area1 + area2 - inter
        return inter / union if union > 0 else 0

    def detect(self, image_path):
        img_rgb = cv2.imread(image_path)
        img_name = os.path.basename(image_path)
        gray_image_path = os.path.join(self.gray_images_path, img_name)
        img_gray = cv2.imread(gray_image_path) if os.path.exists(gray_image_path) else None

        rgb_normal_detections = self.run_with_rotations(img_rgb, self.rgb_model, 'rgb_normal')
        rgb_normal_clean = self.nms(rgb_normal_detections)

        img_normalized, _, _ = self.normalizer.normalize_and_detect(image_path, self.rgb_model, self.conf_thresh, self.iou_thresh)
        rgb_bright_detections = self.run_with_rotations(img_normalized, self.rgb_model, 'rgb_bright')
        rgb_bright_clean = self.nms(rgb_bright_detections)

        gray_clean = []
        if img_gray is not None:
            gray_detections = self.run_with_rotations(img_gray, self.gray_model, 'gray')
            gray_clean = self.nms(gray_detections)

        all_detections = rgb_normal_clean + rgb_bright_clean + gray_clean
        final_trees = self.ensemble_vote(all_detections)
        return final_trees

    def ensemble_vote(self, all_detections):
        groups = []
        used = [False] * len(all_detections)

        for i in range(len(all_detections)):
            if used[i]:
                continue
            group = [all_detections[i]]
            used[i] = True
            for j in range(i+1, len(all_detections)):
                if used[j]:
                    continue
                for g in group:
                    if self.calculate_iou(all_detections[j]['box'], g['box']) > self.iou_grouping_thresh:
                        group.append(all_detections[j])
                        used[j] = True
                        break
            groups.append(group)

        final_detections = []
        for group in groups:
            models_in_group = set([d['model'] for d in group])
            num_models = len(models_in_group)
            weighted_sum = sum(d['confidence'] * self.weights[d['model']] for d in group)
            weight_sum = sum(self.weights[d['model']] for d in group)
            weighted_confidence = weighted_sum / weight_sum

            keep = False
            if num_models >= 3 and weighted_confidence > 0.4:
                keep = True
            elif num_models == 2 and weighted_confidence > 0.6:
                keep = True
            elif num_models == 1 and weighted_confidence > 0.7:
                keep = True

            if keep:
                final_detections.append(self.merge_group(group))
        return final_detections

    def merge_group(self, group):
        total_weight = 0
        x1_w = y1_w = x2_w = y2_w = 0
        for det in group:
            w = det['confidence'] * self.weights[det['model']]
            x1_w += det['box'][0] * w
            y1_w += det['box'][1] * w
            x2_w += det['box'][2] * w
            y2_w += det['box'][3] * w
            total_weight += w
        return {
            'box': [x1_w/total_weight, y1_w/total_weight, x2_w/total_weight, y2_w/total_weight],
            'confidence': sum(d['confidence'] for d in group) / len(group)
        }


# ==================== EVALUATION FUNCTIONS ====================
def load_gt_boxes(label_path, img_w, img_h):
    boxes = []
    if not label_path.exists():
        return np.array([]).reshape(0, 4)
    for line in label_path.read_text().strip().split('\n'):
        if not line.strip():
            continue
        parts = line.strip().split()
        if len(parts) >= 5:
            xc, yc, w, h = map(float, parts[1:5])
            boxes.append([(xc - w/2) * img_w, (yc - h/2) * img_h, (xc + w/2) * img_w, (yc + h/2) * img_h])
    return np.array(boxes) if boxes else np.array([]).reshape(0, 4)

def compute_iou(b1, b2):
    x1, y1 = max(b1[0], b2[0]), max(b1[1], b2[1])
    x2, y2 = min(b1[2], b2[2]), min(b1[3], b2[3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    union = (b1[2]-b1[0])*(b1[3]-b1[1]) + (b2[2]-b2[0])*(b2[3]-b2[1]) - inter
    return inter/union if union > 0 else 0

def compute_ap(all_preds, total_gt, iou_thresh):
    if not all_preds or total_gt == 0:
        return 0.0
    all_preds_sorted = sorted(all_preds, key=lambda x: -x['confidence'])
    tp = np.zeros(len(all_preds_sorted))
    fp = np.zeros(len(all_preds_sorted))
    gt_matched = {}

    for i, pred in enumerate(all_preds_sorted):
        img_id = pred['img_id']
        gt_boxes = pred['gt_boxes']
        pred_box = pred['box']

        if img_id not in gt_matched:
            gt_matched[img_id] = set()

        best_iou = 0
        best_gt_idx = -1

        for gt_idx, gt_box in enumerate(gt_boxes):
            if gt_idx in gt_matched[img_id]:
                continue
            iou = compute_iou(pred_box, gt_box)
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = gt_idx

        if best_iou >= iou_thresh and best_gt_idx >= 0:
            tp[i] = 1
            gt_matched[img_id].add(best_gt_idx)
        else:
            fp[i] = 1

    tp_cumsum = np.cumsum(tp)
    fp_cumsum = np.cumsum(fp)
    precision = tp_cumsum / (tp_cumsum + fp_cumsum + 1e-10)
    recall = tp_cumsum / (total_gt + 1e-10)
    precision = np.concatenate([[1], precision, [0]])
    recall = np.concatenate([[0], recall, [recall[-1] if len(recall) > 0 else 0]])

    for i in range(len(precision) - 2, -1, -1):
        precision[i] = max(precision[i], precision[i + 1])

    indices = np.where(recall[1:] != recall[:-1])[0] + 1
    ap = np.sum((recall[indices] - recall[indices - 1]) * precision[indices])
    return ap


# ==================== MAIN EVALUATION ====================
print("Initializing ensemble detector...")
ensemble = EnsembleTreeDetector(
    rgb_model_path=RGB_MODEL_PATH,
    gray_model_path=GRAY_MODEL_PATH,
    gray_images_path=GRAY_IMAGES_DIR
)

print(f"\nEvaluating on {len(images)} images...\n")

all_predictions = []
total_gt_boxes = 0

for i, img_path in enumerate(images):
    final_trees = ensemble.detect(str(img_path))
    img = cv2.imread(str(img_path))
    img_h, img_w = img.shape[:2]
    gt_boxes = load_gt_boxes(Path(LABELS_DIR) / (img_path.stem + ".txt"), img_w, img_h)
    total_gt_boxes += len(gt_boxes)

    for tree in final_trees:
        all_predictions.append({
            'box': tree['box'], 'confidence': tree['confidence'],
            'img_id': img_path.stem, 'gt_boxes': gt_boxes
        })

    if (i+1) % 10 == 0:
        print(f"Processed {i+1}/{len(images)}")

print("\nComputing mAP@50...")
ap_50 = compute_ap(all_predictions, total_gt_boxes, iou_thresh=0.5)

print("Computing mAP@50-95...")
iou_thresholds = np.arange(0.5, 1.0, 0.05)
aps = []
for iou_t in iou_thresholds:
    ap = compute_ap(all_predictions, total_gt_boxes, iou_thresh=iou_t)
    aps.append(ap)
    print(f"  AP@{iou_t:.2f}: {ap:.4f}")

map_50_95 = np.mean(aps)

tp, fp, fn = 0, 0, 0
for img_path in images:
    img = cv2.imread(str(img_path))
    img_h, img_w = img.shape[:2]
    gt_boxes = load_gt_boxes(Path(LABELS_DIR) / (img_path.stem + ".txt"), img_w, img_h)

    preds_for_img = [p for p in all_predictions if p['img_id'] == img_path.stem]
    pred_boxes = np.array([p['box'] for p in preds_for_img]) if preds_for_img else np.array([]).reshape(0,4)
    pred_confs = np.array([p['confidence'] for p in preds_for_img]) if preds_for_img else np.array([])

    matched_gt = set()
    for idx in np.argsort(-pred_confs) if len(pred_confs) else []:
        pred = pred_boxes[idx]
        best_iou, best_gt = 0, -1
        for gi, gt in enumerate(gt_boxes):
            if gi in matched_gt:
                continue
            iou = compute_iou(pred, gt)
            if iou > best_iou:
                best_iou, best_gt = iou, gi
        if best_iou >= 0.5:
            tp += 1
            matched_gt.add(best_gt)
        else:
            fp += 1
    fn += len(gt_boxes) - len(matched_gt)

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"\n{'='*50}")
print(f"RESULTS - Ensemble")
print(f"{'='*50}")
print(f"Precision:    {precision:.4f}")
print(f"Recall:       {recall:.4f}")
print(f"F1 Score:     {f1:.4f}")
print(f"mAP@50:       {ap_50:.4f}")
print(f"mAP@50-95:    {map_50_95:.4f}")
print(f"{'='*50}")
print(f"Total GT:     {total_gt_boxes}")
print(f"Total Preds:  {len(all_predictions)}")
print(f"{'='*50}")

# Cleanup
shutil.rmtree(GRAY_IMAGES_DIR)
print("Cleaned up temp files")

Creating greyscale images...
Created 59 greyscale images

Initializing ensemble detector...
Loading models...

Evaluating on 59 images...

Processed 10/59
Processed 20/59
Processed 30/59
Processed 40/59
Processed 50/59

Computing mAP@50...
Computing mAP@50-95...
  AP@0.50: 0.9104
  AP@0.55: 0.8211
  AP@0.60: 0.6678
  AP@0.65: 0.4602
  AP@0.70: 0.2518
  AP@0.75: 0.1008
  AP@0.80: 0.0249
  AP@0.85: 0.0036
  AP@0.90: 0.0001
  AP@0.95: 0.0000

RESULTS - Ensemble
Precision:    0.8993
Recall:       0.9612
F1 Score:     0.9292
mAP@50:       0.9104
mAP@50-95:    0.3241
Total GT:     26747
Total Preds:  28588
Cleaned up temp files


# Usage Examples

RGB Model


In [ ]:
from ultralytics import YOLO

# ==================== CONFIGURATION ====================
# Update this path to your base directory
BASE_DIR = "Object Detection Pipeline"

# Update this to any image you want to test
IMAGE_PATH = "path/to/your/image.jpg"

# Paths (no need to edit)
MODEL_PATH = f"{BASE_DIR}/Models/orange_tree_model_350images/tree_training/weights/best.pt"

# ==================== INFERENCE ====================
# Load model
model = YOLO(MODEL_PATH)

# Run detection
results = model(IMAGE_PATH, conf=0.35, iou=0.45, max_det=10000)

# Show results
print(f"Detected {len(results[0].boxes)} trees")
results[0].show()  # or results[0].save("output_rgb.jpg")


image 1/1 C:\Users\afifb\OneDrive\Documents\mech year 2\research\Paper Dr.Bilal\pipeline\object detection\Object Detection Pipeline\test\images\DJI_20250203143256_0008_V_JPG_jpg.rf.50a4a527bf93928cbd5770f99e633407.jpg: 640x640 537 trees, 84.2ms
Speed: 3.0ms preprocess, 84.2ms inference, 21.4ms postprocess per image at shape (1, 3, 640, 640)
Detected 537 trees


Greyscale Model

In [ ]:
from ultralytics import YOLO
import cv2

# ==================== CONFIGURATION ====================
# Update this path to your base directory
BASE_DIR = "Object Detection Pipeline"

# Update this to any image you want to test
IMAGE_PATH = "path/to/your/image.jpg"

# Paths (no need to edit)
MODEL_PATH = f"{BASE_DIR}/Models/orange_tree_model_350images_greyscale/tree_training/weights/best.pt"

# ==================== INFERENCE ====================
# Load model
model = YOLO(MODEL_PATH)

# Convert image to greyscale
img = cv2.imread(IMAGE_PATH)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
gray_3ch = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
cv2.imwrite("temp_gray.jpg", gray_3ch)

# Run detection
results = model("temp_gray.jpg", conf=0.35, iou=0.45, max_det=10000)

# Show results
print(f"Detected {len(results[0].boxes)} trees")
results[0].show()


image 1/1 c:\Users\afifb\OneDrive\Documents\mech year 2\research\Paper Dr.Bilal\pipeline\object detection\temp_gray.jpg: 640x640 526 trees, 280.6ms
Speed: 2.6ms preprocess, 280.6ms inference, 1095.7ms postprocess per image at shape (1, 3, 640, 640)
Detected 526 trees


Brightness Normalized

In [ ]:
import os
import cv2
import torch

# ==================== CONFIGURATION ====================
# Update this path to your base directory
BASE_DIR = "Object Detection Pipeline"

# Update this to any image you want to test
IMAGE_PATH = "path/to/your/image.jpg"

# Paths (no need to edit)
RGB_MODEL_PATH = f"{BASE_DIR}/Models/orange_tree_model_350images/tree_training/weights/best.pt"
GRAY_MODEL_PATH = f"{BASE_DIR}/Models/orange_tree_model_350images_greyscale/tree_training/weights/best.pt"
GRAY_TEMP_DIR = "temp_gray/"

# ==================== INFERENCE ====================
# Clear GPU cache
torch.cuda.empty_cache()

os.makedirs(GRAY_TEMP_DIR, exist_ok=True)

# Create grayscale version
img = cv2.imread(IMAGE_PATH)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
gray_3ch = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
gray_path = os.path.join(GRAY_TEMP_DIR, os.path.basename(IMAGE_PATH))
cv2.imwrite(gray_path, gray_3ch)

# Load ensemble (EnsembleTreeDetector class must be defined above)
ensemble = EnsembleTreeDetector(
    rgb_model_path=RGB_MODEL_PATH,
    gray_model_path=GRAY_MODEL_PATH,
    gray_images_path=GRAY_TEMP_DIR
)

# Run detection
trees, stats = ensemble.detect(IMAGE_PATH)

# Show results
print(f"Detected {len(trees)} trees")

Detected 538 trees


Ensemble

In [ ]:
import os
import cv2
import torch

# ==================== CONFIGURATION ====================
# Update this path to your base directory
BASE_DIR = "Object Detection Pipeline"

# Update this to any image you want to test
IMAGE_PATH = "path/to/your/image.jpg"

# Paths (no need to edit)
RGB_MODEL_PATH = f"{BASE_DIR}/Models/orange_tree_model_350images/tree_training/weights/best.pt"
GRAY_MODEL_PATH = f"{BASE_DIR}/Models/orange_tree_model_350images_greyscale/tree_training/weights/best.pt"
GRAY_TEMP_DIR = "temp_gray/"

# ==================== INFERENCE ====================
# Clear GPU cache
torch.cuda.empty_cache()

os.makedirs(GRAY_TEMP_DIR, exist_ok=True)

# Create grayscale version
img = cv2.imread(IMAGE_PATH)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
gray_3ch = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
gray_path = os.path.join(GRAY_TEMP_DIR, os.path.basename(IMAGE_PATH))
cv2.imwrite(gray_path, gray_3ch)

# Load ensemble (EnsembleTreeDetector class must be defined above)
ensemble = EnsembleTreeDetector(
    rgb_model_path=RGB_MODEL_PATH,
    gray_model_path=GRAY_MODEL_PATH,
    gray_images_path=GRAY_TEMP_DIR
)

# Run detection
trees, stats = ensemble.detect(IMAGE_PATH)

# Show results
print(f"Detected {len(trees)} trees")

Loading models...

Processing: DJI_20250203143256_0008_V_JPG_jpg.rf.50a4a527bf93928cbd5770f99e633407.jpg
  Running RGB normal...
    Found 598 trees
  Running RGB brightness normalized...
    Found 593 trees
  Running Grayscale...
    Found 530 trees
  Performing ensemble voting...
  Final result: 573 trees detected
Detected 573 trees
